# Runwatch local simulation

This simulation is intentionally local: it creates small files below `.runwatch/simulation-data`, updates a launcher-owned localhost dashboard, emits Runwatch progress and resource events, and makes no external network or AWS calls.

## Configure a unique simulation run

In [ ]:
import json
import os
import time
from pathlib import Path

In [ ]:
simulation_id = time.strftime("%Y%m%d-%H%M%S")
batch_count = int(os.environ.get("RUNWATCH_SIMULATION_BATCHES", "20"))
batch_delay_seconds = float(os.environ.get("RUNWATCH_SIMULATION_DELAY_SECONDS", "0.5"))
dashboard_url = os.environ["RUNWATCH_SIMULATION_DASHBOARD_URL"]
dashboard_status_path = Path(os.environ["RUNWATCH_SIMULATION_DASHBOARD_STATUS_PATH"])
if batch_count < 1 or batch_delay_seconds < 0:
    raise ValueError("simulation batches must be positive and delay nonnegative")
output_dir = Path(".runwatch") / "simulation-data" / simulation_id
parts_dir = output_dir / "parts"
log_path = output_dir / "worker.log"
parts_dir.mkdir(parents=True)
dashboard_status_path.write_text(
    json.dumps({"message": "Simulation starting", "metrics": {"completed batches": 0}}),
    encoding="utf-8",
)
print(
    f"Simulation artifacts: {output_dir} | "
    f"batches={batch_count} delay={batch_delay_seconds:g}s"
)

## Register local monitors

In [ ]:
from runwatch import emit_progress, local

In [ ]:
local.emit_system_metrics(
    include_host=True,
    include_kernel=True,
    gpu="all",
    logical_key="simulation-system",
    poll_interval_seconds=0.5,
)
local.emit_file_count(
    parts_dir,
    pattern="*.json",
    logical_key="simulation-parts",
    expected_count=batch_count,
    blocking=True,
    poll_interval_seconds=0.5,
)
local.emit_line_count(
    log_path,
    logical_key="simulation-log",
    expected_lines=batch_count * 5,
    tail_lines=12,
    blocking=True,
    poll_interval_seconds=0.5,
)
local.emit_dashboard(
    dashboard_url,
    name="Simulation results",
    health_path="/status.json",
    logical_key="simulation-dashboard",
    poll_interval_seconds=0.5,
)

## Run the simulated workload

In [ ]:
batch_results: list[dict[str, float | int]] = []
processed_rows = 0

for batch in range(1, batch_count + 1):
    started = time.perf_counter()
    checksum = sum((item * batch) % 104_729 for item in range(300_000))
    rows = 2_000 + 125 * batch
    elapsed_seconds = time.perf_counter() - started
    processed_rows += rows
    result = {
        "batch": batch,
        "rows": rows,
        "checksum": checksum,
        "seconds": elapsed_seconds,
        "throughput": rows / elapsed_seconds,
    }
    (parts_dir / f"part-{batch:02d}.json").write_text(
        json.dumps(result) + "\n",
        encoding="utf-8",
    )
    with log_path.open("a", encoding="utf-8") as log:
        for shard in range(1, 6):
            log.write(
                f"batch={batch:02d} shard={shard} rows={rows // 5} "
                f"checksum={checksum}\n"
            )
    batch_results.append(result)
    dashboard_status = {
        "message": f"Completed batch {batch} of {batch_count}",
        "metrics": {
            "completed_batches": batch,
            "total_batches": batch_count,
            "processed_rows": f"{processed_rows:,}",
            "latest_throughput": f"{result['throughput']:,.0f} rows/s",
        },
    }
    dashboard_status_tmp = dashboard_status_path.with_suffix(".tmp")
    dashboard_status_tmp.write_text(json.dumps(dashboard_status), encoding="utf-8")
    dashboard_status_tmp.replace(dashboard_status_path)
    emit_progress(
        batch,
        total=batch_count,
        unit="batches",
        message=f"Completed batch {batch} of {batch_count}",
        metrics={"rows": processed_rows, "throughput": result["throughput"]},
    )
    print(f"batch {batch:03d}/{batch_count} | rows={processed_rows:,}")
    time.sleep(batch_delay_seconds)

## Inspect results

In [ ]:
results = batch_results

In [ ]:
print(json.dumps(results[-8:], indent=2))

In [ ]:
print(f"Completed {len(results)} batches")

In [ ]:
summary = {
    "simulation_id": simulation_id,
    "configured_batches": batch_count,
    "batch_delay_seconds": batch_delay_seconds,
    "output_directory": str(output_dir),
    "batches": len(results),
    "rows": sum(int(result["rows"]) for result in results),
    "part_files": len(list(parts_dir.glob("*.json"))),
    "log_lines": len(log_path.read_text(encoding="utf-8").splitlines()),
    "mean_throughput": sum(float(result["throughput"]) for result in results)
    / len(results),
}
dashboard_status = {
    "message": "Simulation completed",
    "metrics": {
        "completed_batches": len(results),
        "processed_rows": f"{processed_rows:,}",
        "part_files": summary["part_files"],
        "log_lines": summary["log_lines"],
    },
}
dashboard_status_tmp = dashboard_status_path.with_suffix(".tmp")
dashboard_status_tmp.write_text(json.dumps(dashboard_status), encoding="utf-8")
dashboard_status_tmp.replace(dashboard_status_path)
print(json.dumps(summary, indent=2))